# Synthetic Data Validation

This notebook checks whether the synthetic alternative-financial behavior data is plausible enough for controlled experiments. It does not train a model.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.synthetic_generator import SyntheticDataConfig, save_synthetic_dataset
from src.data.validation import validate_alternative_features, validate_financial_history

sns.set_theme(style='whitegrid')

In [ ]:
config = SyntheticDataConfig(n_applicants=10_000, months=12, seed=42)
profiles, history, features, metadata = save_synthetic_dataset(PROJECT_ROOT, config)

validate_financial_history(history)
validate_alternative_features(features)

profiles.shape, history.shape, features.shape, metadata

In [ ]:
profiles['population_group'].value_counts(normalize=True).sort_index()

In [ ]:
history[['monthly_income', 'monthly_expenses', 'savings_balance', 'rent_amount', 'utility_bill_amount', 'transaction_count', 'net_cashflow']].describe().T

In [ ]:
plot_data = history.merge(profiles[['applicant_id', 'population_group']], on='applicant_id', suffixes=('', '_profile'))
plot_data['expense_income_ratio'] = plot_data['monthly_expenses'] / plot_data['monthly_income']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(data=plot_data, x='monthly_income', hue='population_group', bins=50, element='step', stat='density', common_norm=False, ax=axes[0, 0])
sns.histplot(data=plot_data, x='expense_income_ratio', hue='population_group', bins=40, element='step', stat='density', common_norm=False, ax=axes[0, 1])
sns.boxplot(data=plot_data, x='population_group', y='savings_balance', ax=axes[1, 0])
sns.boxplot(data=plot_data, x='population_group', y='transaction_count', ax=axes[1, 1])
fig.tight_layout()

In [ ]:
payment_summary = plot_data.groupby('population_group').agg(
    utility_on_time_rate=('utility_payment_status', lambda s: (s == 'on_time').mean()),
    rent_on_time_rate=('rent_payment_status', lambda s: (s == 'on_time').mean()),
    positive_cashflow_rate=('net_cashflow', lambda s: (s > 0).mean()),
    cashflow_volatility=('net_cashflow', 'std'),
    transaction_frequency=('transaction_count', 'mean'),
)
payment_summary

In [ ]:
features_with_group = profiles[['applicant_id', 'population_group']].merge(features, on='applicant_id')
features_with_group.groupby('population_group').mean(numeric_only=True).T

In [ ]:
selected_features = [
    'monthly_income_mean',
    'income_variability',
    'expense_to_income_ratio',
    'savings_consistency',
    'utility_payment_regularity',
    'rent_payment_regularity',
    'cashflow_volatility',
    'transaction_frequency',
    'financial_buffer_ratio',
]

plt.figure(figsize=(10, 8))
sns.heatmap(features[selected_features].corr(), cmap='vlag', center=0, annot=False)
plt.title('Correlation Between Derived Alternative-Financial Features')
plt.tight_layout()

## Validation Notes

- Inspect whether the three groups overlap rather than separating perfectly.
- Check that income, expenses, rent, utilities, balances, and transaction counts stay in plausible ranges.
- Check that payment regularity and cashflow volatility vary across groups without encoding a future model outcome.
- Treat any scientific conclusions as provisional because this is synthetic data only.